# Advanced Python `__slots__`: 18 problems with complete solutions

**Level:** intermediate → advanced · **Runtime:** Python 3.10+ · **Format:** problem → worked solution → executable assertions.

## Learning goals

- Explain instance slot storage, member descriptors, class dictionaries, and the real limits of `__slots__`.
- Design validated, slotted objects; work with inheritance, mixins, weak references, descriptors, and cached properties.
- Integrate slots with dataclasses, copying, pickling, introspection, and realistic application code.
- Measure memory and lookup performance responsibly instead of assuming that slots are always faster.

**How to use:** Run all cells in order. Read a problem before scrolling to its solution, implement it independently if desired, and then run the corresponding tests. Problems use distinct class names to avoid accidental notebook-state collisions. Tests use plain `assert` and an explicit exception helper; they require no third-party packages.

**Important correction to the supplied example:** Mumbai's latitude is approximately **19.0760° N** and longitude **72.8777° E**. When an initializer takes `(name, longitude, latitude)`, call it with `('Mumbai', 72.8777, 19.0760)`—not the reverse. The notebook uses named arguments where practical.

**Precise mental model:** Declaring slots normally installs descriptors in the **class** dictionary and suppresses the **instance** dictionary only if no base/subclass introduces one. Slots are not immutability, type checking, security, or a guaranteed speedup. Class attributes remain independently modifiable unless deliberately restricted.

## Setup: dependencies, assertion helper, and environment

In [1]:
from __future__ import annotations

import copy
import dataclasses
import gc
import math
import pickle
import statistics
import sys
import timeit
import tracemalloc
import weakref
from dataclasses import dataclass, field
from functools import cached_property
from typing import ClassVar


def expect_raises(error_type, function, *args, **kwargs):
    """Check that *function* raises the specified exception class."""
    try:
        function(*args, **kwargs)
    except error_type as exc:
        return exc
    except Exception as exc:
        raise AssertionError(
            f"Expected {error_type}, got {type(exc).__name__}: {exc}"
        ) from exc
    raise AssertionError(f"Expected {error_type}, but no exception was raised")


print(f"Python: {sys.version.split()[0]} | implementation: {sys.implementation.name}")
assert sys.version_info >= (3, 10), "This notebook requires Python 3.10+."

Python: 3.13.7 | implementation: cpython


---
## Problem 1 — Repair and harden the `Location` example

**Task.** Implement a location with the exact instance slots `name`, `_longitude`, and `_latitude`. Validate coordinates (`longitude ∈ [-180, 180]`, `latitude ∈ [-90, 90]`), exclude booleans and non-finite numbers, provide read-only public coordinate properties, and make instances reject unexpected attributes. Demonstrate that adding a **class** attribute still works. Explicitly fix the Mumbai coordinate ordering.

**Design:** `__slots__` limits attribute names; validation and the read-only API are separate decisions. The private-by-convention fields can still be reassigned directly; do not confuse this design with a frozen object.

### Solution 1

In [2]:
class Location:
    __slots__ = ("name", "_longitude", "_latitude")

    def __init__(self, name: str, longitude: float, latitude: float) -> None:
        if not isinstance(name, str) or not name.strip():
            raise ValueError("name must be a non-empty string")
        self.name = name
        self._longitude = self._coordinate(longitude, -180.0, 180.0, "longitude")
        self._latitude = self._coordinate(latitude, -90.0, 90.0, "latitude")

    @staticmethod
    def _coordinate(value: float, low: float, high: float, label: str) -> float:
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError(f"{label} must be a real number, excluding bool")
        result = float(value)
        if not math.isfinite(result) or not low <= result <= high:
            raise ValueError(f"{label} must be finite and between {low} and {high}")
        return result

    @property
    def longitude(self) -> float:
        return self._longitude

    @property
    def latitude(self) -> float:
        return self._latitude

    def __repr__(self) -> str:
        return (
            f"Location(name={self.name!r}, longitude={self.longitude!r}, "
            f"latitude={self.latitude!r})"
        )


mumbai = Location("Mumbai", longitude=72.8777, latitude=19.0760)
assert (mumbai.longitude, mumbai.latitude) == (72.8777, 19.076)
assert not hasattr(mumbai, "__dict__")
assert "name" in Location.__dict__  # Member descriptor in the class mapping proxy.
assert "_longitude" in Location.__dict__
expect_raises(AttributeError, setattr, mumbai, "map_link", "https://example.com")
expect_raises(AttributeError, setattr, mumbai, "latitude", 0.0)
expect_raises(ValueError, Location, "Bad", longitude=181, latitude=0)
expect_raises(ValueError, Location, "Bad", longitude=0, latitude=float("nan"))
expect_raises(TypeError, Location, "Bad", longitude=True, latitude=0)

Location.map_service = "example mapping provider"  # Changes the class, not a slot.
assert Location.map_service == "example mapping provider"
assert mumbai.map_service == Location.map_service
print(mumbai)
print("Problem 1: all checks passed")

Location(name='Mumbai', longitude=72.8777, latitude=19.076)
Problem 1: all checks passed


---
## Problem 2 — Deletion, uninitialized slots, and descriptor shadowing

**Task.** Demonstrate: (a) an allowed slot may be deleted and reassigned; (b) a slot exists in the class before it has a value in a newly constructed instance; (c) replacing a slot's class-level descriptor with an ordinary class value is a **bad idea** and can make stored data inaccessible through normal lookup.

**Hint:** Inspect `Demo.__dict__["value"]` before experimenting. Isolate the destructive descriptor-replacement demonstration in a disposable class.

### Solution 2

In [3]:
class DeletableSlot:
    __slots__ = ("value",)

    def __init__(self, value):
        self.value = value


item = DeletableSlot(10)
del item.value
expect_raises(AttributeError, getattr, item, "value")
item.value = 20
assert item.value == 20

raw = object.__new__(DeletableSlot)  # __init__ has NOT been called.
expect_raises(AttributeError, getattr, raw, "value")
raw.value = "now initialized"
assert raw.value == "now initialized"


class ShadowedSlot:
    __slots__ = ("value",)

    def __init__(self, value):
        self.value = value


shadowed = ShadowedSlot(7)
saved_descriptor = ShadowedSlot.__dict__["value"]
assert saved_descriptor.__get__(shadowed, ShadowedSlot) == 7
ShadowedSlot.value = "class-level replacement"  # Intentionally BAD practice.
assert shadowed.value == "class-level replacement"
assert saved_descriptor.__get__(shadowed, ShadowedSlot) == 7
expect_raises(AttributeError, setattr, shadowed, "value", 8)
# Restore the descriptor to avoid leaving the demonstration in a broken state.
ShadowedSlot.value = saved_descriptor
assert shadowed.value == 7
print("Problem 2: all checks passed")

Problem 2: all checks passed


**Why it works:** Slot declarations create class-level descriptors. The descriptor mediates per-instance storage. Overwriting it does not delete the underlying stored value, but it breaks normal access to that slot. Always initialize defaults in `__init__` (or a dataclass `field`), not by overwriting the descriptor.

---
## Problem 3 — Enforce a validation invariant with a custom descriptor

**Task.** Create a `PositiveFinite` data descriptor and a slotted `InvoiceLine` whose `unit_price` and `quantity` are validated on **every public assignment**. Make deletion of these public properties fail. Use distinct backing slots, so the descriptor names never collide with generated slot descriptors.

**Edge cases:** Reject booleans, zero, negative values, infinity, and NaN. Show explicitly that direct access to underscore-prefixed storage can bypass the public interface: slots do **not** create a security boundary.

### Solution 3

In [4]:
class PositiveFinite:
    """Data descriptor storing a strictly positive finite float in a backing slot."""

    def __init__(self, storage_name: str):
        self.storage_name = storage_name
        self.public_name = None

    def __set_name__(self, owner, name):
        self.public_name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return getattr(instance, self.storage_name)

    def __set__(self, instance, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError(f"{self.public_name} must be a number, excluding bool")
        converted = float(value)
        if not math.isfinite(converted) or converted <= 0:
            raise ValueError(f"{self.public_name} must be positive and finite")
        setattr(instance, self.storage_name, converted)

    def __delete__(self, instance):
        raise AttributeError(f"cannot delete {self.public_name}")


class InvoiceLine:
    __slots__ = ("sku", "_unit_price", "_quantity")
    unit_price = PositiveFinite("_unit_price")
    quantity = PositiveFinite("_quantity")

    def __init__(self, sku: str, unit_price: float, quantity: float):
        if not isinstance(sku, str) or not sku.strip():
            raise ValueError("sku must be a non-empty string")
        self.sku = sku
        self.unit_price = unit_price
        self.quantity = quantity

    @property
    def total(self) -> float:
        return self.unit_price * self.quantity


line = InvoiceLine("P-001", 12.5, 4)
assert line.total == 50.0
line.quantity = 5
assert line.total == 62.5
for invalid in (0, -1, float("inf"), float("nan")):
    expect_raises(ValueError, setattr, line, "quantity", invalid)
expect_raises(TypeError, setattr, line, "quantity", True)
expect_raises(AttributeError, delattr, line, "unit_price")
expect_raises(AttributeError, setattr, line, "extra", 123)
assert not hasattr(line, "__dict__")

# This is intentionally possible: an underscore is a convention, not access control.
line._quantity = -100.0
assert line.total == -1250.0
line.quantity = 5  # Restore the invariant via the supported API.
print("Problem 3: all checks passed")

Problem 3: all checks passed


---
## Problem 4 — The inheritance trap: slots do not automatically propagate

**Task.** Build four variants: slotted parent; child with no declared `__slots__`; child with an additional slot; and child explicitly allowing a dictionary. Determine which instances permit unexpected attributes. Also show why an empty `__slots__ = ()` is useful in subclasses that add no fields.

### Solution 4

In [5]:
class SlottedParent:
    __slots__ = ("identifier",)

    def __init__(self, identifier):
        self.identifier = identifier


class AccidentalDictChild(SlottedParent):
    pass  # No __slots__: ordinary subclass instances acquire __dict__.


class FullySlottedChild(SlottedParent):
    __slots__ = ("label",)

    def __init__(self, identifier, label):
        super().__init__(identifier)
        self.label = label


class NoNewFieldsChild(SlottedParent):
    __slots__ = ()


class IntentionallyDynamicChild(SlottedParent):
    __slots__ = ("__dict__",)


parent = SlottedParent(1)
accidental = AccidentalDictChild(2)
fully = FullySlottedChild(3, "three")
no_new = NoNewFieldsChild(4)
dynamic = IntentionallyDynamicChild(5)

assert not hasattr(parent, "__dict__")
assert hasattr(accidental, "__dict__")
assert not hasattr(fully, "__dict__")
assert not hasattr(no_new, "__dict__")
assert hasattr(dynamic, "__dict__")
accidental.extra = "permitted"
dynamic.extra = "intentionally permitted"
expect_raises(AttributeError, setattr, fully, "extra", "denied")
expect_raises(AttributeError, setattr, no_new, "extra", "denied")
assert fully.identifier == 3 and fully.label == "three"
assert FullySlottedChild.__slots__ == ("label",)  # Inherited slots aren't repeated.
print("Problem 4: all checks passed")

Problem 4: all checks passed


---
## Problem 5 — Multiple inheritance, layout conflicts, and safe mixins

**Task.** Prove that two unrelated parents with nonempty slots can create a layout conflict. Then design a working mixin with `__slots__ = ()` that adds behavior but does not demand a separate slot layout. Verify that a slotted concrete subclass has no instance dictionary.

**Rule of thumb:** In normal multiple inheritance, at most one independent parent should contribute nonempty instance slot storage. A shared compatible slotted ancestor and empty-slot mixins can work; arbitrary combinations cannot.

### Solution 5

In [6]:
class LeftLayout:
    __slots__ = ("left",)


class RightLayout:
    __slots__ = ("right",)


# Creating the class, rather than instantiating it, raises TypeError.
expect_raises(
    TypeError,
    type,
    "ConflictingLayout",
    (LeftLayout, RightLayout),
    {"__slots__": ()},
)


class PrintableMixin:
    __slots__ = ()

    def describe(self) -> str:
        return f"{type(self).__name__}(identifier={self.identifier!r})"


class Ticket(SlottedParent, PrintableMixin):
    __slots__ = ("priority",)

    def __init__(self, identifier: int, priority: int):
        super().__init__(identifier)
        self.priority = priority


ticket = Ticket(42, 3)
assert ticket.describe() == "Ticket(identifier=42)"
assert ticket.priority == 3
assert not hasattr(ticket, "__dict__")
expect_raises(AttributeError, setattr, ticket, "surprise", True)
print("Problem 5: all checks passed")

Problem 5: all checks passed


---
## Problem 6 — Private-slot name mangling is not privacy

**Task.** Declare a slot starting with two underscores, read it through a public property, inspect the generated mangled descriptor name, and demonstrate that outside code can still access it. Avoid treating name mangling as encryption or strong encapsulation.

### Solution 6

In [7]:
class ApiCredential:
    __slots__ = ("__token",)

    def __init__(self, token: str):
        self.__token = token

    @property
    def token_length(self) -> int:
        return len(self.__token)


credential = ApiCredential("illustrative-token")
assert credential.token_length == len("illustrative-token")
assert "_ApiCredential__token" in ApiCredential.__dict__
expect_raises(AttributeError, getattr, credential, "__token")
assert credential._ApiCredential__token == "illustrative-token"  # Accessible!
assert not hasattr(credential, "__dict__")
print("Problem 6: all checks passed")

Problem 6: all checks passed


---
## Problem 7 — Weak references and the `__weakref__` slot

**Task.** Explain why an otherwise ordinary slotted class cannot necessarily be weak-referenced. Add exactly the slot needed to support `weakref.ref`; verify that a weak-value cache accepts its instances. Compare with an ordinary dict-backed class.

**Caution:** `__weakref__` is not a user payload field. Declare it in a slotted hierarchy only once when needed; do not redeclare a slot supplied by a parent.

### Solution 7

In [8]:
class NoWeakRef:
    __slots__ = ("key",)

    def __init__(self, key):
        self.key = key


class WeakRefEnabled:
    __slots__ = ("key", "__weakref__")

    def __init__(self, key):
        self.key = key


class OrdinaryObject:
    def __init__(self, key):
        self.key = key


not_weak = NoWeakRef("a")
expect_raises(TypeError, weakref.ref, not_weak)
weak_object = WeakRefEnabled("b")
reference = weakref.ref(weak_object)
assert reference() is weak_object
ordinary = OrdinaryObject("c")
assert weakref.ref(ordinary)() is ordinary

cache = weakref.WeakValueDictionary()
cache["b"] = weak_object
assert cache["b"] is weak_object
assert not hasattr(weak_object, "__dict__")
print("Problem 7: all checks passed")

Problem 7: all checks passed


---
## Problem 8 — Benchmark memory without misleading `getsizeof` claims

**Task.** Compare a dict-backed and a slotted class with the same three fields. Report (1) their shallow instance sizes, (2) the additional shallow size of the normal object's instance dictionary, and (3) the approximate *difference in allocations* for a collection of many live instances using `tracemalloc`.

**Methodology:** Warm up both classes first; ensure objects remain alive during measurement; run each population separately after garbage collection; report observations without asserting that slots must win across Python implementations/builds. `sys.getsizeof` measures only shallow sizes, not the transitive size of the object graph. `tracemalloc` records tracked Python allocations, not total process RSS or every native allocation.

### Solution 8

In [9]:
class RegularTriplet:
    def __init__(self, x, y, z):
        self.x, self.y, self.z = x, y, z


class SlottedTriplet:
    __slots__ = ("x", "y", "z")

    def __init__(self, x, y, z):
        self.x, self.y, self.z = x, y, z


regular_example = RegularTriplet(1, 2, 3)
slotted_example = SlottedTriplet(1, 2, 3)
print("Regular shallow object:", sys.getsizeof(regular_example), "bytes")
print("Regular __dict__ (additional shallow):", sys.getsizeof(regular_example.__dict__), "bytes")
print("Slotted shallow object:", sys.getsizeof(slotted_example), "bytes")


def tracked_population_size(cls, count=20_000):
    """Return traced allocation change while a population remains alive."""
    cls(0, 0, 0)  # Warm up relevant class machinery.
    gc.collect()
    tracemalloc.start()
    try:
        before, _ = tracemalloc.get_traced_memory()
        population = [cls(i, i, i) for i in range(count)]
        after, _ = tracemalloc.get_traced_memory()
        assert len(population) == count
        return after - before
    finally:
        tracemalloc.stop()


regular_bytes = tracked_population_size(RegularTriplet)
slotted_bytes = tracked_population_size(SlottedTriplet)
print(f"Regular: {regular_bytes:,} traced bytes for 20,000 objects")
print(f"Slotted: {slotted_bytes:,} traced bytes for 20,000 objects")
print(f"Observed difference (regular - slotted): {regular_bytes - slotted_bytes:,} bytes")
assert regular_bytes > 0 and slotted_bytes > 0
print("Problem 8: measurement completed; no implementation-dependent ordering asserted")

Regular shallow object: 48 bytes
Regular __dict__ (additional shallow): 296 bytes
Slotted shallow object: 56 bytes
Regular: 2,729,174 traced bytes for 20,000 objects
Slotted: 1,925,456 traced bytes for 20,000 objects
Observed difference (regular - slotted): 803,718 bytes
Problem 8: measurement completed; no implementation-dependent ordering asserted


---
## Problem 9 — Attribute access benchmarks that avoid false promises

**Task.** Benchmark *reads and writes* on preconstructed regular and slotted objects. Repeat measurements and print best and median times. Do not conflate object construction, GC overhead, or unrelated calculations with attribute access; never turn a microbenchmark from one machine into a universal speed guarantee.

**Practice:** Keep functions symmetric, measure both operations, verify correctness first, and use `timeit.repeat`. Numbers are observations only.

### Solution 9

In [10]:
regular_bench = RegularTriplet(1, 2, 3)
slotted_bench = SlottedTriplet(1, 2, 3)


def read_regular():
    return regular_bench.x


def read_slotted():
    return slotted_bench.x


def write_regular():
    regular_bench.x = 99


def write_slotted():
    slotted_bench.x = 99


assert read_regular() == read_slotted() == 1
write_regular()
write_slotted()
assert read_regular() == read_slotted() == 99


def timing_summary(function, repetitions=3, loops=150_000):
    runs = timeit.repeat(function, number=loops, repeat=repetitions)
    return min(runs) / loops * 1e9, statistics.median(runs) / loops * 1e9


for label, function in (
    ("regular read", read_regular),
    ("slotted read", read_slotted),
    ("regular write", write_regular),
    ("slotted write", write_slotted),
):
    best, median = timing_summary(function)
    print(f"{label:15s} best={best:9.1f} ns/op   median={median:9.1f} ns/op")
print("Problem 9: measurements completed; interpret results for this environment only")

regular read    best=     35.9 ns/op   median=     38.3 ns/op
slotted read    best=     54.2 ns/op   median=     55.3 ns/op
regular write   best=     68.5 ns/op   median=     90.8 ns/op
slotted write   best=     50.6 ns/op   median=     57.4 ns/op
Problem 9: measurements completed; interpret results for this environment only


---
## Problem 10 — Slots and dataclasses, including frozen and weak-reference variants

**Task.** Build an equality-aware, hashable, immutable 2D point using `@dataclass(slots=True, frozen=True)`. Demonstrate that generated instances do not have a dictionary. In Python 3.11+, also use `weakref_slot=True`. Explain why a regular `eq=True, frozen=False` dataclass is typically unhashable.

**Version note:** `dataclass(slots=True)` requires Python 3.10+; the `weakref_slot` option requires 3.11+. `frozen=True` protects normal assignment; it is not a tamper-proof security mechanism.

### Solution 10

In [11]:
@dataclass(slots=True, frozen=True)
class Vector2:
    x: float
    y: float

    def magnitude(self) -> float:
        return math.hypot(self.x, self.y)


v = Vector2(3.0, 4.0)
assert v.magnitude() == 5.0
assert v == Vector2(3.0, 4.0)
assert {v: "length five"}[Vector2(3.0, 4.0)] == "length five"
assert not hasattr(v, "__dict__")
expect_raises(dataclasses.FrozenInstanceError, setattr, v, "x", 100)
expect_raises(TypeError, weakref.ref, v)  # No weakref slot was requested.


@dataclass(slots=True)
class MutableVector:
    x: float
    y: float


expect_raises(TypeError, hash, MutableVector(1.0, 2.0))

if sys.version_info >= (3, 11):
    @dataclass(slots=True, frozen=True, weakref_slot=True)
    class WeakVector:
        x: float
        y: float

    weak_vector = WeakVector(1, 2)
    assert weakref.ref(weak_vector)() is weak_vector
    print("weakref_slot=True works (Python 3.11+)")
else:
    print("weakref_slot=True skipped: it requires Python 3.11+")
print("Problem 10: all checks passed")

weakref_slot=True works (Python 3.11+)
Problem 10: all checks passed


---
## Problem 11 — Class variables and slot-name conflicts

**Task.** Show that defining a class variable with the *same name* as a slot fails at class creation time, but a separate class constant is fine. Illustrate how to give slotted instances a default field value in `__init__`. With a dataclass, annotate shared configuration using `ClassVar` so it is not considered a dataclass field.

### Solution 11

In [12]:
def create_invalid_class():
    class InvalidDefault:
        __slots__ = ("count",)
        count = 0  # Conflicts with the generated slot descriptor.
    return InvalidDefault


expect_raises(ValueError, create_invalid_class)


class Counter:
    __slots__ = ("count",)
    DEFAULT_COUNT = 0

    def __init__(self, count=None):
        self.count = self.DEFAULT_COUNT if count is None else count


assert Counter().count == 0
assert Counter(9).count == 9
Counter.DEFAULT_COUNT = 5
assert Counter().count == 5


@dataclass(slots=True)
class Price:
    amount: float
    currency: ClassVar[str] = "USD"


price = Price(10.5)
assert price.currency == "USD"
assert [f.name for f in dataclasses.fields(Price)] == ["amount"]
expect_raises(AttributeError, setattr, price, "unexpected", 1)
print("Problem 11: all checks passed")

Problem 11: all checks passed


---
## Problem 12 — Copy semantics and mutable payloads

**Task.** Use a slotted object containing a list. Explain and test the difference between `copy.copy` and `copy.deepcopy`. Propose a safer immutable representation when a mutable list is not actually needed.

**Note:** Slots restrict which attributes exist; they do not make an attribute's value immutable, and Python's copy machinery can usually copy ordinary slotted classes.

### Solution 12

In [13]:
class TaggedRecord:
    __slots__ = ("identifier", "tags")

    def __init__(self, identifier: str, tags):
        self.identifier = identifier
        self.tags = list(tags)  # Defensive copy of caller-owned input.


original = TaggedRecord("R1", ["new"])
shallow = copy.copy(original)
deep = copy.deepcopy(original)
assert shallow is not original and deep is not original
assert shallow.tags is original.tags  # Shared nested reference.
assert deep.tags is not original.tags
original.tags.append("reviewed")
assert shallow.tags == ["new", "reviewed"]
assert deep.tags == ["new"]


@dataclass(frozen=True, slots=True)
class ImmutableTags:
    identifier: str
    tags: tuple[str, ...]


safer = ImmutableTags("R1", ("new",))
assert isinstance(safer.tags, tuple)
expect_raises(dataclasses.FrozenInstanceError, setattr, safer, "tags", ("changed",))
assert safer.tags == ("new",)
print("Problem 12: all checks passed")

Problem 12: all checks passed


---
## Problem 13 — Pickle, versioned state, and migration

**Task.** Confirm that a plain, top-level slotted class can be pickled. Then implement explicit, versioned state for a `SensorReading`, with validation on load and an example of migrating a legacy version-0 dictionary. Reject unknown future versions.

**Security:** Never unpickle bytes from an untrusted source. Pickle is an object reconstruction format that can execute arbitrary code. Explicit `__getstate__`/`__setstate__` is helpful when you control the format, but does not make unpickling safe.

### Solution 13

In [14]:
triplet = SlottedTriplet(1, 2, 3)
restored_triplet = pickle.loads(pickle.dumps(triplet))
assert (restored_triplet.x, restored_triplet.y, restored_triplet.z) == (1, 2, 3)
assert not hasattr(restored_triplet, "__dict__")


class SensorReading:
    __slots__ = ("sensor_id", "celsius")
    STATE_VERSION = 1

    def __init__(self, sensor_id: str, celsius: float):
        if not isinstance(sensor_id, str) or not sensor_id:
            raise ValueError("sensor_id must be a non-empty string")
        if isinstance(celsius, bool) or not isinstance(celsius, (int, float)):
            raise TypeError("celsius must be numeric")
        celsius = float(celsius)
        if not math.isfinite(celsius) or celsius < -273.15:
            raise ValueError("invalid Celsius reading")
        self.sensor_id = sensor_id
        self.celsius = celsius

    def __getstate__(self):
        return {
            "version": self.STATE_VERSION,
            "sensor_id": self.sensor_id,
            "celsius": self.celsius,
        }

    def __setstate__(self, state):
        version = state.get("version", 0)
        if version == 0:
            # Hypothetical old representation: 'temperature' was Celsius.
            sensor_id = state["sensor"]
            celsius = state["temperature"]
        elif version == 1:
            sensor_id = state["sensor_id"]
            celsius = state["celsius"]
        else:
            raise ValueError(f"unsupported SensorReading state version: {version!r}")
        self.__init__(sensor_id, celsius)


reading = SensorReading("T-01", 21.5)
restored = pickle.loads(pickle.dumps(reading))  # Trusted, locally produced bytes ONLY.
assert (restored.sensor_id, restored.celsius) == ("T-01", 21.5)
assert not hasattr(restored, "__dict__")
legacy = object.__new__(SensorReading)
legacy.__setstate__({"sensor": "OLD", "temperature": 23.0})
assert (legacy.sensor_id, legacy.celsius) == ("OLD", 23.0)
expect_raises(ValueError, legacy.__setstate__, {"version": 999})
expect_raises(ValueError, legacy.__setstate__, {"version": 1, "sensor_id": "X", "celsius": -300})
print("Problem 13: all checks passed")

Problem 13: all checks passed


---
## Problem 14 — Introspection across a slotted class hierarchy

**Task.** Implement `public_state(obj)` that extracts assigned slot values from all classes in the MRO and includes dynamic dictionary values if present. Handle a `__slots__` declaration written as a single string, partially initialized objects, and double-underscore slot-name mangling. Do not use `vars(obj)` blindly on an object that might have no dictionary.

**Scope:** This is a teaching-oriented introspector for ordinary Python slotted classes. It does not promise to capture arbitrary C-extension storage or custom descriptor-managed state; production serialization should use an explicit schema.

### Solution 14

In [15]:
def public_state(obj):
    """Extract ordinary Python slot values plus an optional instance dictionary."""
    result = {}
    # Walk base -> subclass, allowing the most-derived value to replace a duplicate.
    for cls in reversed(type(obj).__mro__):
        slots = cls.__dict__.get("__slots__", ())
        if isinstance(slots, str):
            slots = (slots,)
        # Some third-party classes may define an unusual non-iterable __slots__.
        for declared_name in slots:
            if declared_name in ("__dict__", "__weakref__"):
                continue
            actual_name = declared_name
            if declared_name.startswith("__") and not declared_name.endswith("__"):
                actual_name = f"_{cls.__name__.lstrip('_')}{declared_name}"
            try:
                result[actual_name] = getattr(obj, actual_name)
            except AttributeError:
                pass  # Slot was not assigned (or was explicitly deleted).
    instance_dict = getattr(obj, "__dict__", None)
    if instance_dict is not None:
        result.update(instance_dict)
    return result


assert public_state(mumbai) == {
    "name": "Mumbai", "_longitude": 72.8777, "_latitude": 19.076
}
assert public_state(fully) == {"identifier": 3, "label": "three"}
assert public_state(accidental) == {"identifier": 2, "extra": "permitted"}
assert public_state(credential) == {"_ApiCredential__token": "illustrative-token"}


class OneStringSlot:
    __slots__ = "only"


one = OneStringSlot()
assert public_state(one) == {}  # Not assigned yet.
one.only = 42
assert public_state(one) == {"only": 42}
expect_raises(TypeError, vars, one)
print("Problem 14: all checks passed")

Problem 14: all checks passed


---
## Problem 15 — `cached_property` and a slot-only alternative

**Task.** Show why `functools.cached_property` cannot store its cached result on an instance lacking `__dict__`. Implement a correct single-instance cache using a dedicated slot and sentinel. Also demonstrate the opt-in dictionary workaround and its memory/design tradeoff.

**Caution:** The simple sentinel implementation below is intentionally not a thread-synchronization primitive; use a lock if computing the same value exactly once across threads is required.

### Solution 15

In [16]:
class IncompatibleCache:
    __slots__ = ("raw",)

    def __init__(self, raw):
        self.raw = raw

    @cached_property
    def normalized(self):
        return self.raw.strip().lower()


expect_raises(TypeError, getattr, IncompatibleCache("  HELLO  "), "normalized")
_MISSING = object()


class SlotOnlyCache:
    __slots__ = ("raw", "_normalized", "compute_count")

    def __init__(self, raw: str):
        self.raw = raw
        self._normalized = _MISSING
        self.compute_count = 0

    @property
    def normalized(self) -> str:
        if self._normalized is _MISSING:
            self.compute_count += 1
            self._normalized = self.raw.strip().lower()
        return self._normalized

    def update_raw(self, raw: str) -> None:
        self.raw = raw
        self._normalized = _MISSING  # Explicit cache invalidation.


cached = SlotOnlyCache("  HELLO  ")
assert cached.normalized == "hello"
assert cached.normalized == "hello"
assert cached.compute_count == 1
cached.update_raw(" NEW ")
assert cached.normalized == "new" and cached.compute_count == 2
assert not hasattr(cached, "__dict__")


class DictEnabledCache:
    __slots__ = ("raw", "__dict__")

    def __init__(self, raw):
        self.raw = raw

    @cached_property
    def normalized(self):
        return self.raw.strip().lower()


with_dict = DictEnabledCache("  OK  ")
assert with_dict.normalized == "ok"
assert with_dict.__dict__ == {"normalized": "ok"}
print("Problem 15: all checks passed")

Problem 15: all checks passed


---
## Problem 16 — A weak-key metadata cache without reference leaks

**Task.** Use `WeakKeyDictionary` to associate extra metadata with a slotted object **without** adding dynamic instance attributes. Give the object weak-reference support and identity-based hashing. Confirm that removing the last strong reference allows a cache entry to disappear after collection.

**Pitfalls:** A weak key must support both weak references and hashing. A default mutable dataclass with generated equality may be unhashable. Cache **values** must not retain strong references back to keys if you expect the entries to disappear.

### Solution 16

In [17]:
class Job:
    __slots__ = ("job_id", "__weakref__")

    def __init__(self, job_id: str):
        self.job_id = job_id


metadata = weakref.WeakKeyDictionary()
job = Job("J-01")
metadata[job] = {"attempts": 2}  # No back-reference to job.
assert metadata[job]["attempts"] == 2
assert not hasattr(job, "__dict__")
assert len(metadata) == 1
reference = weakref.ref(job)
del job
gc.collect()
assert reference() is None
assert len(metadata) == 0
print("Problem 16: all checks passed")

Problem 16: all checks passed


---
## Problem 17 — Cooperative slotted inheritance and avoiding redundant slots

**Task.** Build a shared slotted base and two descendant levels that use cooperative `super()`. Each class should declare only its *new* storage names. Compare the actual MRO and verify that no instance dictionary is introduced. Explain why repeating an inherited slot name in a subclass is not a sensible extension strategy.

**Best practice:** Slot declarations describe storage added by *that class*, not the complete inventory of all ancestors. Re-declaring a parent's slot name can hide the inherited descriptor and make behavior confusing; never rely on it.

### Solution 17

In [18]:
class BaseEvent:
    __slots__ = ("event_id",)

    def __init__(self, event_id: str, **kwargs):
        super().__init__(**kwargs)
        self.event_id = event_id


class TimedEvent(BaseEvent):
    __slots__ = ("timestamp",)

    def __init__(self, *, timestamp: float, **kwargs):
        super().__init__(**kwargs)
        self.timestamp = timestamp


class AuditEvent(TimedEvent):
    __slots__ = ("actor",)

    def __init__(self, *, actor: str, **kwargs):
        super().__init__(**kwargs)
        self.actor = actor


event = AuditEvent(event_id="E-1", timestamp=123.5, actor="analyst")
assert (event.event_id, event.timestamp, event.actor) == ("E-1", 123.5, "analyst")
assert not hasattr(event, "__dict__")
assert AuditEvent.__slots__ == ("actor",)
assert TimedEvent.__slots__ == ("timestamp",)
assert [cls.__name__ for cls in AuditEvent.__mro__] == [
    "AuditEvent", "TimedEvent", "BaseEvent", "object"
]
expect_raises(AttributeError, setattr, event, "unplanned", "field")
print("Problem 17: all checks passed")

Problem 17: all checks passed


---
## Problem 18 — Capstone: immutable geographic records and a nearby-location query

**Task.** Implement a small, validated geospatial catalog using slotted, frozen dataclasses. Coordinates must be correctly labeled and validated; record names must be nonblank; the catalog must keep an immutable tuple of records; distance calculations must use the haversine formula; queries must reject invalid limits and return a deterministic distance-then-name ordering. Test exact matches, empty catalogs, negative limits, immutability, and the Mumbai coordinate correction.

**Constraints and tradeoffs:** Coordinates are `(latitude, longitude)` inside `GeoPoint`, explicitly differing from the earlier `Location(name, longitude, latitude)` API. Haversine gives approximate great-circle distance for a spherical Earth (radius 6,371.0088 km); do not treat it as survey-grade geodesy or a scalable spatial index. A linear scan is suitable for this small teaching catalog.

### Solution 18 — Model and validation

In [19]:
EARTH_RADIUS_KM = 6371.0088


@dataclass(frozen=True, slots=True)
class GeoPoint:
    latitude: float
    longitude: float

    def __post_init__(self):
        lat = Location._coordinate(self.latitude, -90.0, 90.0, "latitude")
        lon = Location._coordinate(self.longitude, -180.0, 180.0, "longitude")
        object.__setattr__(self, "latitude", lat)
        object.__setattr__(self, "longitude", lon)

    def distance_km(self, other: GeoPoint) -> float:
        if not isinstance(other, GeoPoint):
            raise TypeError("other must be a GeoPoint")
        lat1, lon1 = map(math.radians, (self.latitude, self.longitude))
        lat2, lon2 = map(math.radians, (other.latitude, other.longitude))
        delta_lat = lat2 - lat1
        delta_lon = lon2 - lon1
        a = (
            math.sin(delta_lat / 2) ** 2
            + math.cos(lat1) * math.cos(lat2) * math.sin(delta_lon / 2) ** 2
        )
        # Clamp against floating-point drift near identical/antipodal points.
        a = min(1.0, max(0.0, a))
        return 2 * EARTH_RADIUS_KM * math.asin(math.sqrt(a))


@dataclass(frozen=True, slots=True)
class GeoRecord:
    name: str
    point: GeoPoint

    def __post_init__(self):
        if not isinstance(self.name, str) or not self.name.strip():
            raise ValueError("record name must be a non-empty string")
        if not isinstance(self.point, GeoPoint):
            raise TypeError("point must be a GeoPoint")


mumbai_point = GeoPoint(latitude=19.0760, longitude=72.8777)
pune_point = GeoPoint(latitude=18.5204, longitude=73.8567)
london_point = GeoPoint(latitude=51.5074, longitude=-0.1278)
assert mumbai_point.distance_km(mumbai_point) == 0.0
assert abs(mumbai_point.distance_km(pune_point) - 120) < 30
assert math.isclose(
    mumbai_point.distance_km(pune_point),
    pune_point.distance_km(mumbai_point),
    rel_tol=1e-12,
)
expect_raises(ValueError, GeoPoint, latitude=91, longitude=0)
expect_raises(ValueError, GeoPoint, latitude=0, longitude=float("inf"))
expect_raises(dataclasses.FrozenInstanceError, setattr, mumbai_point, "latitude", 0)
assert not hasattr(mumbai_point, "__dict__")
print("Capstone model checks passed")

Capstone model checks passed


### Solution 18 — Catalog, queries, and edge-case tests

In [20]:
class GeoCatalog:
    __slots__ = ("_records",)

    def __init__(self, records):
        records = tuple(records)  # Detach from mutable caller-owned containers.
        if not all(isinstance(record, GeoRecord) for record in records):
            raise TypeError("each catalog entry must be a GeoRecord")
        self._records = records

    @property
    def records(self) -> tuple[GeoRecord, ...]:
        return self._records

    def nearby(self, origin: GeoPoint, *, limit: int = 3):
        """Return tuple[(record, km), ...] ordered by distance then name."""
        if not isinstance(origin, GeoPoint):
            raise TypeError("origin must be a GeoPoint")
        if isinstance(limit, bool) or not isinstance(limit, int) or limit < 0:
            raise ValueError("limit must be a non-negative integer, excluding bool")
        if limit == 0:
            return ()
        measured = [(record, origin.distance_km(record.point)) for record in self._records]
        measured.sort(key=lambda pair: (pair[1], pair[0].name))
        return tuple(measured[:limit])


entries = [
    GeoRecord("Mumbai", mumbai_point),
    GeoRecord("Pune", pune_point),
    GeoRecord("London", london_point),
]
catalog = GeoCatalog(entries)
entries.clear()  # Catalog has safely retained its own immutable tuple.
assert len(catalog.records) == 3
nearest = catalog.nearby(mumbai_point, limit=2)
assert [record.name for record, distance in nearest] == ["Mumbai", "Pune"]
assert nearest[0][1] == 0.0
assert nearest[1][1] > 0
assert catalog.nearby(mumbai_point, limit=0) == ()
assert GeoCatalog([]).nearby(mumbai_point) == ()
expect_raises(ValueError, catalog.nearby, mumbai_point, limit=-1)
expect_raises(ValueError, catalog.nearby, mumbai_point, limit=True)
expect_raises(TypeError, catalog.nearby, "not a point")
expect_raises(TypeError, GeoCatalog, ["not a record"])
expect_raises(AttributeError, setattr, catalog, "unplanned", 1)
expect_raises(dataclasses.FrozenInstanceError, setattr, catalog.records[0], "name", "Edited")

# Deterministic tie-breaker: equal coordinates yield equal distance.
tied = GeoCatalog([
    GeoRecord("Zulu", mumbai_point), GeoRecord("Alpha", mumbai_point)
])
assert [rec.name for rec, _ in tied.nearby(mumbai_point)] == ["Alpha", "Zulu"]
print("Nearest locations:")
for record, kilometers in nearest:
    print(f"  {record.name:8s} {kilometers:9.2f} km")
print("Problem 18: all checks passed")

Nearest locations:
  Mumbai        0.00 km
  Pune        120.15 km
Problem 18: all checks passed


---
## Final checkpoint — cumulative contract tests

The following cell checks the most important claims across the notebook in one place. Run the notebook **from the first cell** to avoid dependence on stale interactive definitions.

In [21]:
assert Location.__slots__ == ("name", "_longitude", "_latitude")
assert not hasattr(Location("Mumbai", 72.8777, 19.076), "__dict__")
assert hasattr(AccidentalDictChild(1), "__dict__")
assert not hasattr(FullySlottedChild(1, "a"), "__dict__")
weak_check = WeakRefEnabled("x")
assert weakref.ref(weak_check)() is weak_check
assert hash(Vector2(1, 2)) == hash(Vector2(1, 2))
assert len(catalog.nearby(mumbai_point, limit=99)) == 3
assert mumbai_point.distance_km(mumbai_point) == 0.0
assert len(metadata) == 0
print("ALL 18 PROBLEMS COMPLETED: cumulative checks passed")

ALL 18 PROBLEMS COMPLETED: cumulative checks passed


## Takeaways / decision checklist

1. Use `__slots__` when you have a stable attribute schema and enough instances to justify measuring memory. Prefer clarity over speculative micro-optimization.
2. Validate data independently; slots do not validate types or values, prevent deletion, or guarantee immutability.
3. Verify the **whole inheritance chain**: one subclass without slots may restore `__dict__`; multiple independent slotted layouts may conflict.
4. Explicitly request `__weakref__` when needed; `__dict__` is an opt-in if dynamic attributes or standard `cached_property` are essential.
5. For simple value objects on Python 3.10+, consider `@dataclass(slots=True, frozen=True)`; use `weakref_slot=True` on Python 3.11+ if appropriate.
6. For caching, copying, serialization, and introspection, test the actual behavior rather than assuming dictionary-backed conventions.
7. A memory benchmark must keep its objects alive, use like-for-like classes, and describe what its measuring tool actually measures. A single timeit result never proves a universal speed advantage.

**Further independent practice:** Extend the capstone with bounding-box filtering; add a JSON-safe `to_dict`/`from_dict` format for `GeoRecord`; build a factory generating 100,000 records and repeat the memory benchmark with the two alternative record designs. Test correctness first and report environment, methodology, and uncertainty with any benchmark numbers.